# 实验二：输入电流
固定其他 LIF 参数，比较恒定电流与临界发放、ISI 和平均频率。统计覆盖 2000 ms，膜电位图展示前 200 ms。

In [ ]:
import csv
import json
import sys
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

root = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lif/model.py").is_file()
)
sys.path.insert(0, str(root))
from lif.model import SimulationParameters, simulate_experiment  # noqa: E402

baseline = SimulationParameters()
out = root / "lif/output/experiment2"
out.mkdir(parents=True, exist_ok=True)
levels = [0.5, 1.0, 1.5, 1.6, 2.0, 2.5, 3.0]
rows, traces = [], []
fig, axes = plt.subplots(2, 1, figsize=(9, 7), layout="constrained")
for current in levels:
    data = simulate_experiment(replace(baseline, current=current))
    traces.append(data)
    times = data["time_ms"][data["spikes"].astype(bool)]
    count = len(times)
    rows.append(
        {
            "current_nA": current,
            "spike_count": count,
            "average_rate_Hz": count * 1000 / baseline.duration,
            "first_spike_ms": float(times[0]) if count else None,
            "mean_ISI_ms": float(np.diff(times).mean()) if count > 1 else None,
        }
    )
    if current in (1.0, 1.5, 2.0, 3.0):
        axes[0].plot(data["time_ms"], data["voltage"], label=f"{current:g} nA")
    if current <= 1.5:
        assert count == 0
axes[0].axhline(baseline.threshold, color="black", linestyle=":", label="Threshold")
axes[0].set(xlim=(0, 200), xlabel="Time (ms)", ylabel="Voltage (mV)")
axes[0].legend(ncol=3)
axes[1].plot(levels, [r["average_rate_Hz"] for r in rows], "o-", label="Simulation")
critical = (baseline.threshold - baseline.rest) / baseline.resistance
axes[1].axvline(critical, color="black", linestyle=":", label="Critical current")
dense = np.linspace(critical + 0.001, max(levels), 300)
period = baseline.refractory - baseline.tau * np.log1p(-critical / dense)
axes[1].plot(dense, 1000 / period, "--", label="Steady-state theory")
axes[1].set(xlabel="Current (nA)", ylabel="Firing rate (Hz)")
axes[1].legend()
fig.savefig(out / "current_comparison.png", dpi=160)
with (out / "summary.csv").open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0])
    writer.writeheader()
    writer.writerows(rows)
(out / "parameters.json").write_text(
    json.dumps({"baseline": asdict(baseline), "currents_nA": levels}, indent=2),
    encoding="utf-8",
)
np.savez_compressed(
    out / "traces.npz",
    current_nA=levels,
    time_ms=traces[0]["time_ms"],
    voltage_mV=np.stack([d["voltage"] for d in traces]),
    spikes=np.stack([d["spikes"] for d in traces]),
)
for row in rows:
    print(row)
plt.show()